# 🚗 AI Insurance Claim Processing System Using Multi-Agentic AI

A **stateful multi-agent system** built with **LangGraph** that automates the initial review of
a motor insurance claim: it reads uploaded documents, checks them against the policy, assesses
fraud/risk, evaluates the claimed amount, and produces a recommendation.

**This system gives a RECOMMENDATION only — it is NOT a legally binding insurance decision.**
A human claims handler always makes the final call.

## Business Problem

Insurance companies receive a large volume of claims. Employees must manually verify documents,
policy details, claim information, and fraud indicators, which causes:
- Slow claim processing
- High manual workload
- Human errors
- Difficulty identifying suspicious claims
- Delays for genuine customers

## Architecture

```
START -> Claim Intake Agent -> Document Processing Agent -> Policy Verification Agent
         -> Fraud/Risk Agent -> Claim Assessment Agent -> Decision Agent -> END
```

This is a **sequential LangGraph pipeline** — simpler than a fan-out/fan-in design, because each
insurance agent genuinely needs the previous agent's output (e.g. Fraud/Risk needs the Policy
Verification result). Every node reads from and writes to a single shared `TypedDict` state,
the same pattern used across LangGraph agent workflows.

In production, `Document Processing` would call real OCR/Document-AI MCP tools, and
`Policy Verification` could query a policy database via an MCP server. Here we use `pypdf` for
PDF text extraction and the LLM's own vision capability for images, to keep the project simple
and dependency-light.

## 0. Setup

In [ ]:
# Run once
# !pip install langgraph langchain-openai langchain-core python-dotenv pypdf

In [ ]:
import os
import json
import io
from datetime import datetime
from typing import TypedDict, List, Dict, Any

from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

In [ ]:
load_dotenv()   # expects OPENAI_API_KEY in your .env

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — put it in your .env"
print("API key loaded ✅")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Shared Agent State

Every node reads from and writes to this single `TypedDict`, exactly like the reference
property-search project's `PropertyState`.

In [ ]:
class ClaimState(TypedDict):
    claimant_info: dict
    policy_info: dict
    claim_description: str
    claim_amount: float
    claim_type: str
    documents: List[Dict[str, Any]]   # [{"name":..., "type": "pdf"/"image", "bytes": ...}]

    intake_summary: Dict[str, Any]
    extracted_documents: Dict[str, Any]
    policy_verification: Dict[str, Any]
    fraud_risk: Dict[str, Any]
    claim_assessment: Dict[str, Any]
    decision: Dict[str, Any]

    errors: List[str]

## 2. Document Reading Helper

Stand-in for a real Document Processing MCP tool. We use `pypdf` for PDF text extraction. For
images, in the full Streamlit app we call the LLM's vision capability directly — here in the
notebook we keep the demo focused on PDF documents for simplicity.

In [ ]:
def read_pdf_text(file_bytes: bytes) -> str:
    """Extract plain text from a PDF using pypdf."""
    try:
        reader = PdfReader(io.BytesIO(file_bytes))
        text = ""
        for page in reader.pages:
            text += (page.extract_text() or "") + "\n"
        return text.strip()
    except Exception as e:
        return f"(could not read PDF: {e})"

## 3. Agent 1 — Claim Intake Agent
Organizes the raw form inputs into a clean structured summary. Pure Python, no LLM call needed —
same style as the simple state-setup nodes in the reference project.

In [ ]:
def claim_intake_agent(state: ClaimState) -> dict:
    errors = []
    if not state.get("claim_description", "").strip():
        errors.append("Claim description is empty.")
    if not state.get("documents"):
        errors.append("No supporting documents were uploaded.")
    if state.get("claim_amount", 0) <= 0:
        errors.append("Claim amount must be greater than zero.")

    summary = {
        "claimant_name": state["claimant_info"].get("name", "Unknown"),
        "policy_number": state["policy_info"].get("policy_number", "Unknown"),
        "claim_type": state.get("claim_type", "Motor Insurance"),
        "claim_amount": state.get("claim_amount", 0),
        "num_documents": len(state.get("documents", [])),
        "submitted_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    return {"intake_summary": summary, "errors": errors}

## 4. Agent 2 — Document Processing Agent
Reads every uploaded PDF document and stores extracted text.

In [ ]:
def document_processing_agent(state: ClaimState) -> dict:
    if state.get("errors"):
        return {}

    extracted = {}
    for doc in state["documents"]:
        name = doc["name"]
        if doc["type"] == "pdf":
            raw_text = read_pdf_text(doc["bytes"])
            extracted[name] = {"kind": "pdf", "content": raw_text[:3000]}
        else:
            extracted[name] = {"kind": "image", "content": "(image processing demoed in app.py)"}
    return {"extracted_documents": extracted}

## 5. Agent 3 — Policy Verification Agent
Checks whether the claim looks consistent with the uploaded policy document.

In [ ]:
POLICY_SYSTEM_PROMPT = """You are a Policy Verification agent for a motor insurance company.
Given the policy document text and the claim details, check whether the claim appears
consistent with the policy (coverage type, validity dates, insured vehicle, exclusions).

Return ONLY a JSON object (no markdown fences, no extra text) with keys:
  "policy_matches_claim" (true/false/"unclear"),
  "coverage_notes" (short string),
  "concerns" (list of short strings, empty list if none)
"""

def policy_verification_agent(state: ClaimState) -> dict:
    if state.get("errors"):
        return {}

    policy_text = ""
    for name, doc in state.get("extracted_documents", {}).items():
        if "policy" in name.lower():
            policy_text += doc["content"] + "\n"

    if not policy_text.strip():
        return {"policy_verification": {
            "policy_matches_claim": "unclear",
            "coverage_notes": "No policy document was uploaded or text could not be read.",
            "concerns": ["Policy document missing or unreadable."],
        }}

    resp = llm.invoke([
        SystemMessage(content=POLICY_SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Policy document text:\n{policy_text[:3000]}\n\n"
            f"Claim details:\n{json.dumps(state['intake_summary'])}\n"
            f"Claim description: {state['claim_description']}"
        )),
    ])
    raw = resp.content.strip().strip("`").replace("json\n", "", 1)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"policy_matches_claim": "unclear", "coverage_notes": raw[:300], "concerns": []}
    return {"policy_verification": result}

## 6. Agent 4 — Fraud/Risk Agent
Rule-based checks (deterministic, cheap — same style as the reference project's verification
agent) combined with an LLM risk narrative.

In [ ]:
FRAUD_SYSTEM_PROMPT = """You are a Fraud/Risk assessment agent for a motor insurance company.
Given the claim details, extracted documents, policy verification result, and a list of
rule-based flags already detected, write a short risk assessment.

Return ONLY a JSON object (no markdown fences, no extra text) with keys:
  "risk_level" (one of "Low","Medium","High"),
  "risk_score" (integer 0-100),
  "reasoning" (short string, 2-3 sentences)
"""

def fraud_risk_agent(state: ClaimState) -> dict:
    if state.get("errors"):
        return {}

    flags = []
    claim_amount = state.get("claim_amount", 0)
    if claim_amount > 500000:
        flags.append("unusually_high_claim_amount")

    policy_ver = state.get("policy_verification", {})
    if policy_ver.get("policy_matches_claim") is False:
        flags.append("policy_mismatch")
    if policy_ver.get("concerns"):
        flags.append("policy_concerns_present")

    num_docs = state["intake_summary"].get("num_documents", 0)
    if num_docs < 2:
        flags.append("insufficient_supporting_documents")

    resp = llm.invoke([
        SystemMessage(content=FRAUD_SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Claim details: {json.dumps(state['intake_summary'])}\n"
            f"Claim description: {state['claim_description']}\n"
            f"Policy verification: {json.dumps(policy_ver)}\n"
            f"Rule-based flags already detected: {flags}"
        )),
    ])
    raw = resp.content.strip().strip("`").replace("json\n", "", 1)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"risk_level": "Medium", "risk_score": 50, "reasoning": raw[:300]}
    result["rule_based_flags"] = flags
    return {"fraud_risk": result}

## 7. Agent 5 — Claim Assessment Agent
Summarizes the estimated claim impact using the repair estimate/invoice against the claimed
amount.

In [ ]:
ASSESSMENT_SYSTEM_PROMPT = """You are a Claim Assessment agent for a motor insurance company.
Given the claim amount, claim description, and any repair estimate/invoice document text,
summarize the estimated claim impact.

Return ONLY a JSON object (no markdown fences, no extra text) with keys:
  "estimated_amount_reasonable" (true/false/"unclear"),
  "summary" (short string, 2-3 sentences)
"""

def claim_assessment_agent(state: ClaimState) -> dict:
    if state.get("errors"):
        return {}

    estimate_text = ""
    for name, doc in state.get("extracted_documents", {}).items():
        lname = name.lower()
        if "estimate" in lname or "invoice" in lname or "bill" in lname:
            estimate_text += f"[{name}]\n{doc['content']}\n"

    resp = llm.invoke([
        SystemMessage(content=ASSESSMENT_SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Claimed amount: {state.get('claim_amount')}\n"
            f"Claim description: {state['claim_description']}\n"
            f"Repair estimate / invoice text (if any):\n{estimate_text[:3000] or 'None provided'}"
        )),
    ])
    raw = resp.content.strip().strip("`").replace("json\n", "", 1)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"estimated_amount_reasonable": "unclear", "summary": raw[:300]}
    return {"claim_assessment": result}

## 8. Agent 6 — Decision Agent
Combines everything into a final APPROVE / REJECT / MANUAL REVIEW recommendation with reasoning.
This is a recommendation only — a human claims handler makes the final decision.

In [ ]:
DECISION_SYSTEM_PROMPT = """You are the final Decision agent of an AI-assisted insurance claim
processing system. You give a RECOMMENDATION only -- a human claims handler makes the final,
legally binding decision.

Based on the policy verification, fraud/risk assessment, and claim assessment provided,
recommend one of: "APPROVE", "REJECT", "MANUAL REVIEW".

Guidance:
- If risk is High, or policy clearly does not match, or documents are missing -> MANUAL REVIEW or REJECT
- If everything looks consistent and risk is Low -> APPROVE
- When uncertain, prefer MANUAL REVIEW over guessing

Return ONLY a JSON object (no markdown fences, no extra text) with keys:
  "recommendation" (one of "APPROVE","REJECT","MANUAL REVIEW"),
  "explanation" (string, 3-5 sentences explaining the reasoning in plain language)
"""

def decision_agent(state: ClaimState) -> dict:
    if state.get("errors"):
        return {"decision": {
            "recommendation": "MANUAL REVIEW",
            "explanation": "Claim could not be fully processed due to input errors: "
                            + "; ".join(state["errors"]),
        }}

    resp = llm.invoke([
        SystemMessage(content=DECISION_SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Claim summary: {json.dumps(state['intake_summary'])}\n"
            f"Policy verification: {json.dumps(state.get('policy_verification', {}))}\n"
            f"Fraud/Risk assessment: {json.dumps(state.get('fraud_risk', {}))}\n"
            f"Claim assessment: {json.dumps(state.get('claim_assessment', {}))}"
        )),
    ])
    raw = resp.content.strip().strip("`").replace("json\n", "", 1)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"recommendation": "MANUAL REVIEW", "explanation": raw[:500]}
    return {"decision": result}

## 9. Build & Compile the Graph

In [ ]:
graph = StateGraph(ClaimState)

graph.add_node("claim_intake", claim_intake_agent)
graph.add_node("document_processing", document_processing_agent)
graph.add_node("policy_verification", policy_verification_agent)
graph.add_node("fraud_risk", fraud_risk_agent)
graph.add_node("claim_assessment", claim_assessment_agent)
graph.add_node("decision", decision_agent)

graph.add_edge(START, "claim_intake")
graph.add_edge("claim_intake", "document_processing")
graph.add_edge("document_processing", "policy_verification")
graph.add_edge("policy_verification", "fraud_risk")
graph.add_edge("fraud_risk", "claim_assessment")
graph.add_edge("claim_assessment", "decision")
graph.add_edge("decision", END)

claim_agent = graph.compile()
claim_agent

## 10. Example: Process a Genuine Claim

Uses the sample documents in `sample_documents/` (generate them first by running
`python generate_samples.py` from the project root).

In [ ]:
def load_pdf_bytes(path):
    with open(path, "rb") as f:
        return f.read()

genuine_claim_state = {
    "claimant_info": {"name": "Rahul Sharma", "contact": "9876543210"},
    "policy_info": {"policy_number": "POL-2026-88213", "vehicle_number": "DL8CAF1234"},
    "claim_description": "My car was hit from behind while parked outside my office on 12 Aug 2026. "
                          "The rear bumper and tail lamp were damaged.",
    "claim_amount": 23000,
    "claim_type": "Accident Damage",
    "documents": [
        {"name": "sample_policy_genuine.pdf", "type": "pdf",
         "bytes": load_pdf_bytes("sample_documents/sample_policy_genuine.pdf")},
        {"name": "sample_repair_estimate.pdf", "type": "pdf",
         "bytes": load_pdf_bytes("sample_documents/sample_repair_estimate.pdf")},
        {"name": "sample_invoice.pdf", "type": "pdf",
         "bytes": load_pdf_bytes("sample_documents/sample_invoice.pdf")},
    ],
    "errors": [],
}

result = claim_agent.invoke(genuine_claim_state)

print("--- Policy Verification ---")
print(json.dumps(result["policy_verification"], indent=2))
print("\n--- Fraud/Risk ---")
print(json.dumps(result["fraud_risk"], indent=2))
print("\n--- Claim Assessment ---")
print(json.dumps(result["claim_assessment"], indent=2))
print("\n--- Final Recommendation ---")
print(result["decision"]["recommendation"])
print(result["decision"]["explanation"])

## 11. Example: Process a Suspicious / Fraud-Risk Claim

This time the vehicle number on the policy does not match the claim, the policy is
third-party-only (no own-damage cover), and the policy has already expired.

In [ ]:
suspicious_claim_state = {
    "claimant_info": {"name": "Vikram Mehta", "contact": "9123456780"},
    "policy_info": {"policy_number": "POL-2024-11209", "vehicle_number": "HR26XY5678"},
    "claim_description": "My car's engine and interior were badly damaged in an accident. "
                          "Requesting full repair cost.",
    "claim_amount": 650000,
    "claim_type": "Accident Damage",
    "documents": [
        {"name": "sample_policy_suspicious.pdf", "type": "pdf",
         "bytes": load_pdf_bytes("sample_documents/sample_policy_suspicious.pdf")},
    ],
    "errors": [],
}

result2 = claim_agent.invoke(suspicious_claim_state)

print("--- Fraud/Risk ---")
print(json.dumps(result2["fraud_risk"], indent=2))
print("\n--- Final Recommendation ---")
print(result2["decision"]["recommendation"])
print(result2["decision"]["explanation"])

## 12. Example: Missing / Invalid Documents

No documents uploaded at all — the Claim Intake Agent should catch this immediately and the
Decision Agent should recommend MANUAL REVIEW without wasting further LLM calls.

In [ ]:
missing_docs_state = {
    "claimant_info": {"name": "Anita Kumar", "contact": "9012345678"},
    "policy_info": {"policy_number": "POL-2026-55001", "vehicle_number": "KA05MZ9999"},
    "claim_description": "Car was scratched in a parking lot.",
    "claim_amount": 5000,
    "claim_type": "Accident Damage",
    "documents": [],
    "errors": [],
}

result3 = claim_agent.invoke(missing_docs_state)

print("Errors detected:", result3["errors"])
print("\n--- Final Recommendation ---")
print(result3["decision"]["recommendation"])
print(result3["decision"]["explanation"])

## 13. Notes / Production Extensions

* **Document Processing MCP tools**: swap `pypdf` text extraction for a real OCR/Document-AI MCP
  server for scanned/handwritten documents.
* **Policy database**: instead of reading the policy PDF each time, the Policy Verification Agent
  could query a real policy database via an MCP server.
* **Human-in-the-loop**: for claims recommended as MANUAL REVIEW, route to a claims handler's
  queue (e.g. via a database or ticketing system) instead of ending the graph.
* **Persistence**: add a database if claim history / audit trail needs to be stored long-term —
  not required for this first version, which is intentionally kept simple and session-based.
* **Full Streamlit app**: see `app.py` in this project for the complete UI with file upload,
  live agent status, and a downloadable report — including image (accident photo) handling via
  the LLM's vision capability.